<a href="https://colab.research.google.com/github/ekaratnida/Data-Engineer/blob/main/de/bdi-mu/RDD_Version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pyspark.sql import SparkSession

# สร้าง SparkSession
spark = SparkSession.builder \
    .appName("DataEng Workshop") \
    .getOrCreate()

# ทดสอบว่าทำงาน
print("Spark version:", spark.version)

df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .load("sample_data.csv")
df.show(5)

from pyspark.sql.functions import sum, avg

# เลือก column
df.select("name", "amount").show()
# กรองข้อมูล
df.filter(df["amount"] > 1000).show()
# Group by
df.groupBy("category").agg(
      sum("amount").alias("total"),
      avg("amount").alias("avg")
  ).show()
#เพิ่ม column ใหม่
df.withColumn("total", df["price"] * df["qty"]).show()

Spark version: 4.0.2
+--------------+------+-----------+-----+---+
|          name|amount|   category|price|qty|
+--------------+------+-----------+-----+---+
|      Notebook|  1200|Electronics| 1200|  1|
|Wireless Mouse|   450|Electronics|  150|  3|
|    Desk Chair|  2500|  Furniture| 2500|  1|
|    Coffee Mug|   320|    Kitchen|   80|  4|
|  Water Bottle|  1050|    Kitchen|  350|  3|
+--------------+------+-----------+-----+---+
only showing top 5 rows
+--------------+------+
|          name|amount|
+--------------+------+
|      Notebook|  1200|
|Wireless Mouse|   450|
|    Desk Chair|  2500|
|    Coffee Mug|   320|
|  Water Bottle|  1050|
|     Desk Lamp|   850|
+--------------+------+

+------------+------+-----------+-----+---+
|        name|amount|   category|price|qty|
+------------+------+-----------+-----+---+
|    Notebook|  1200|Electronics| 1200|  1|
|  Desk Chair|  2500|  Furniture| 2500|  1|
|Water Bottle|  1050|    Kitchen|  350|  3|
+------------+------+-----------+---

## RDD Version of Data Operations

First, let's get the SparkContext from our existing SparkSession, which is essential for RDD operations.

In [3]:
sc = spark.sparkContext
print("SparkContext obtained.")

SparkContext obtained.


Next, we'll load the `sample_data.csv` file into an RDD. We need to skip the header and then parse each line to extract the data in a usable format (e.g., a dictionary or a tuple with named fields).

In [4]:
data_rdd_raw = sc.textFile("sample_data.csv")

# Skip header
header = data_rdd_raw.first()
data_rdd = data_rdd_raw.filter(lambda line: line != header)

# Define a parsing function to convert each line into a dictionary
def parse_data(line):
    parts = line.split(',')
    return {
        'name': parts[0],
        'amount': int(parts[1]),
        'category': parts[2],
        'price': int(parts[3]),
        'qty': int(parts[4])
    }

# Apply the parsing function to create an RDD of dictionaries
data_parsed_rdd = data_rdd.map(parse_data)

print("Parsed RDD created. Showing first 5 rows:")
for row in data_parsed_rdd.take(5):
    print(row)

Parsed RDD created. Showing first 5 rows:
{'name': 'Notebook', 'amount': 1200, 'category': 'Electronics', 'price': 1200, 'qty': 1}
{'name': 'Wireless Mouse', 'amount': 450, 'category': 'Electronics', 'price': 150, 'qty': 3}
{'name': 'Desk Chair', 'amount': 2500, 'category': 'Furniture', 'price': 2500, 'qty': 1}
{'name': 'Coffee Mug', 'amount': 320, 'category': 'Kitchen', 'price': 80, 'qty': 4}
{'name': 'Water Bottle', 'amount': 1050, 'category': 'Kitchen', 'price': 350, 'qty': 3}


### Select Columns: `name` and `amount`

To replicate `df.select("name", "amount").show()`, we map our RDD to only include these two fields.

In [5]:
selected_rdd = data_parsed_rdd.map(lambda row: {'name': row['name'], 'amount': row['amount']})
print("Selected 'name' and 'amount' columns. Showing all results:")
for row in selected_rdd.collect():
    print(row)

Selected 'name' and 'amount' columns. Showing all results:
{'name': 'Notebook', 'amount': 1200}
{'name': 'Wireless Mouse', 'amount': 450}
{'name': 'Desk Chair', 'amount': 2500}
{'name': 'Coffee Mug', 'amount': 320}
{'name': 'Water Bottle', 'amount': 1050}
{'name': 'Desk Lamp', 'amount': 850}


### Filter Data: `amount > 1000`

To replicate `df.filter(df["amount"] > 1000).show()`, we use the `filter` transformation on the RDD.

In [6]:
filtered_rdd = data_parsed_rdd.filter(lambda row: row['amount'] > 1000)
print("Filtered data where amount > 1000. Showing all results:")
for row in filtered_rdd.collect():
    print(row)

Filtered data where amount > 1000. Showing all results:
{'name': 'Notebook', 'amount': 1200, 'category': 'Electronics', 'price': 1200, 'qty': 1}
{'name': 'Desk Chair', 'amount': 2500, 'category': 'Furniture', 'price': 2500, 'qty': 1}
{'name': 'Water Bottle', 'amount': 1050, 'category': 'Kitchen', 'price': 350, 'qty': 3}


### Group By and Aggregate: `sum` and `avg` of `amount` by `category`

This is more complex with RDDs. We'll use `map` to create key-value pairs, and then `reduceByKey` for the sum and `combineByKey` (or `reduceByKey` followed by a `mapValues`) for the average.

In [7]:
# Calculate sum by category using reduceByKey
sum_by_category_rdd = data_parsed_rdd.map(lambda row: (row['category'], row['amount'])) \
                                   .reduceByKey(lambda a, b: a + b)

# Calculate average by category using combineByKey
# (value, count) -> initial value
# (acc, value) -> merge value into accumulator
# (acc1, acc2) -> merge two accumulators
avg_by_category_rdd = data_parsed_rdd.map(lambda row: (row['category'], (row['amount'], 1))) \
                                   .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
                                   .mapValues(lambda x: x[0] / x[1])

# Join the sum and average RDDs
agg_rdd = sum_by_category_rdd.join(avg_by_category_rdd)

print("Grouped by category with sum and average of amount. Showing all results:")
for category, (total, avg_val) in agg_rdd.collect():
    print(f"Category: {category}, Total Amount: {total}, Average Amount: {avg_val}")

Grouped by category with sum and average of amount. Showing all results:
Category: Furniture, Total Amount: 3350, Average Amount: 1675.0
Category: Electronics, Total Amount: 1650, Average Amount: 825.0
Category: Kitchen, Total Amount: 1370, Average Amount: 685.0


### Add New Column: `total` as `price * qty`

To replicate `df.withColumn("total", df["price"] * df["qty"]).show()`, we'll use a `map` transformation to add a new `total` key to each dictionary in our RDD.

In [8]:
def add_total_column(row):
    new_row = row.copy() # Create a copy to avoid modifying original RDD elements directly
    new_row['total'] = new_row['price'] * new_row['qty']
    return new_row

rdd_with_total = data_parsed_rdd.map(add_total_column)

print("RDD with new 'total' column. Showing all results:")
for row in rdd_with_total.collect():
    print(row)

RDD with new 'total' column. Showing all results:
{'name': 'Notebook', 'amount': 1200, 'category': 'Electronics', 'price': 1200, 'qty': 1, 'total': 1200}
{'name': 'Wireless Mouse', 'amount': 450, 'category': 'Electronics', 'price': 150, 'qty': 3, 'total': 450}
{'name': 'Desk Chair', 'amount': 2500, 'category': 'Furniture', 'price': 2500, 'qty': 1, 'total': 2500}
{'name': 'Coffee Mug', 'amount': 320, 'category': 'Kitchen', 'price': 80, 'qty': 4, 'total': 320}
{'name': 'Water Bottle', 'amount': 1050, 'category': 'Kitchen', 'price': 350, 'qty': 3, 'total': 1050}
{'name': 'Desk Lamp', 'amount': 850, 'category': 'Furniture', 'price': 425, 'qty': 2, 'total': 850}


While DataFrames are generally preferred for most Spark tasks due to their optimizations and ease of use, RDDs (Resilient Distributed Datasets) offer certain benefits, especially in specific scenarios:

1. Low-Level Control and Flexibility: RDDs provide a lower-level API, giving developers fine-grained control over data transformations. This can be beneficial when dealing with very complex or specialized data processing logic that doesn't fit well into the structured operations provided by DataFrames.

2. Unstructured Data: When working with truly unstructured data (e.g., streaming data, custom data formats) where a schema cannot be easily inferred or applied, RDDs might be more suitable. DataFrames excel with structured or semi-structured data.

3. Custom Serialization: RDDs allow you to define custom serialization for your data, which can be useful for performance tuning in some advanced use cases, especially when the default Spark serializers are not optimal.

4. No Schema Overhead: Because RDDs don't enforce a schema, there's no schema inference or management overhead, which can sometimes provide a slight performance edge in very specific, simple operations where schema management would be a bottleneck.

5. Backward Compatibility: RDDs were the original abstraction in Spark, so some legacy codebases or very specialized libraries might still rely on RDDs.

For most modern Spark applications, DataFrames are highly recommended because:

1. Performance Optimizations: DataFrames leverage Spark's Catalyst Optimizer and Tungsten execution engine, providing significant performance benefits through query optimization, code generation, and efficient memory management.

2. Ease of Use: Their structured API, similar to SQL tables, is often more intuitive and easier to work with for data manipulation and analysis.
Schema Enforcement: The schema helps prevent errors and provides better data governance.

3. Interoperability: DataFrames seamlessly integrate with various data sources and allow you to mix SQL, Python, Scala, Java, and R code.
So, while RDDs offer control, DataFrames generally offer better performance and ease of use for the vast majority of tasks involving structured and semi-structured data.